# A4 · Comparación exploratoria V2 frente a V0 · Grupo 15

Se entrenan **base y Top-3**, cada uno desde SciBERT preentrenado con cabeza nueva,
semilla 42 y los mismos hiperparámetros de V0. V2 aísla los documentos en ambos
roles entre train, validación y el test propio. No se reutilizan pesos V0.

**Origen:** training v1 actual y preetiquetas A2; **aislamiento:** V2.
V2 no significa una nueva entrega del dataset ni que el 15% humano esté consolidado.
El test T_B permanece cerrado. El paquete manual de 20 casos no se incorpora aquí.

Un RUN_ID nuevo separa modelos, métricas y checkpoints; MLflow conserva el historial
acumulado en Drive. Ejecutar explicación/código en orden con Colab A100.
El EDA inicial compara los conteos V0/V2. Al final se muestran las validaciones
propias y, por separado, las mismas citas comunes para comparar sin mezclar filas.
Cambian el aislamiento y el tamaño de train: el efecto no se atribuye solo al primero.


In [ ]:
import torch as _v2_torch
if not _v2_torch.cuda.is_available() or "A100" not in _v2_torch.cuda.get_device_name(0):
    raise RuntimeError("Conecta una GPU Colab A100 antes de iniciar esta prueba V2.")
print("GPU V2:", _v2_torch.cuda.get_device_name(0))
import subprocess
import sys

PINNED = [
    "transformers==4.57.1", "accelerate==1.11.0", "datasets==4.3.0",
    "scikit-learn==1.7.2", "matplotlib==3.10.7", "safetensors==0.6.2", "mlflow==3.17.2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED])
print("Instalación terminada. Continúa si estas bibliotecas aún no estaban importadas.")


## 1. Montar Drive y fijar las decisiones del experimento

En Mi unidad ya están el acceso directo **`MAIA_PROYECTO`**, usado como origen de
datos, y la carpeta propia **`MAIA_TESIS`**, usada para las salidas. `DATA_ROOT` apunta
al acceso directo y `RESULTS_ROOT` a la carpeta propia. No se escriben resultados
en el origen compartido. Los lotes se leen de `MAIA_PROYECTO/a2/batches_norw/`.

Copia los archivos pequeños de `TrainingData_Grupo15/` del repositorio a
`MAIA_TESIS/TrainingData_Grupo15/`, directamente en tu carpeta propia, como ya la
subiste a Drive. No hace falta duplicar los lotes de A2.
El manifiesto original conserva sus rutas históricas como procedencia; estas
variables determinan las rutas de esta ejecución.

Los ocho lotes requeridos se comprueban más adelante. Los archivos gigantes de
`inputs/` no son necesarios si los lotes ya contienen los textos de A2.

**Decisiones de esta prueba:** V2, validación
aproximada del 10% agrupada por citante, semilla 42, 4 épocas máximas, LR 2e-5,
batch efectivo 32 y CE sin ponderar. V0 permite compartir citados entre particiones;
V1 separa los citados y V2 también los roles inversos. Se medirá el impacto de las tres.
No se usa un corte temporal: T_B ya está congelado y esta ejecución no mide generalización
temporal. Tampoco se fuerza estratificación por sección; se auditará su distribución.

`RUN_ID` identifica una ejecución: usar un nombre nuevo para un nuevo experimento.
`RESUME=True` permite retomar únicamente checkpoints compatibles de ese directorio.
Los resultados se guardan en Drive privado bajo un RUN_ID V2 nuevo; V0 queda como referencia de lectura.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fuera de Colab: configura rutas locales y REQUIRE_A100=False para pruebas.")

DATA_ROOT = Path("/content/drive/MyDrive/MAIA_PROYECTO")  # acceso directo: solo lectura
RESULTS_ROOT = Path("/content/drive/MyDrive/MAIA_TESIS")  # carpeta propia: resultados
assert DATA_ROOT.is_dir(), f"No se encuentra el acceso directo: {DATA_ROOT}"
assert RESULTS_ROOT.is_dir(), f"No se encuentra la carpeta de resultados: {RESULTS_ROOT}"
if RESULTS_ROOT.resolve().is_relative_to(DATA_ROOT.resolve()):
    raise ValueError("RESULTS_ROOT debe estar fuera del origen compartido DATA_ROOT.")
IDS_DIR = RESULTS_ROOT / "TrainingData_Grupo15"
BATCH_DIR = DATA_ROOT / "a2/batches_norw"
POOL_PATH = DATA_ROOT / "a2/pool_v11.1_ext/prelabels_pool.jsonl"  # opcional
GOLD_FILES = []  # exploración con las mismas preetiquetas A2 de V0; no cargar T_B
GOLD_FINAL = False  # True solo tras confirmar adjudicación y liberaciones cerradas
RUN_TEST = False  # True únicamente para la evaluación final acordada; no para elegir modelos

VARIANT = "V2"  # V0, V1 o V2; conservar el mismo valor para ambas configuraciones
RUN_ID = "g15_a4_v2_seed42_mlflow_01"
OUTPUT_ROOT = RESULTS_ROOT / "a4" / RUN_ID
BASELINE_ROOT = RESULTS_ROOT / "a4/g15_a4_v0_seed42_mlflow_01"
BASELINE_EVALUATION_ROOT = BASELINE_ROOT / "evaluations/v2_fp32_37dc6f909cde"
assert OUTPUT_ROOT != BASELINE_ROOT
CONFIGS = ["base", "fragmentos"]  # ambos modelos se entrenan desde SciBERT preentrenado
SEED = 42
SPLIT_SEED = "G15-A4-split-v1"
VAL_FRAC = 0.10
MAX_LENGTH = 512
CONTEXT_BUDGET = 192
EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH = 16
GRAD_ACCUM = 2
EVAL_BATCH = 32
REQUIRE_A100 = True
RESUME = False
SNAPSHOT_INTERVAL_SECONDS = 300
LABELS = ["Background", "Gap", "Basis", "Comparison", "Application",
          "Improvement/Modification", "Evidence", "Identification of the Originator",
          "Further Reading"]
LABEL2ID = {v: k for k, v in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
assert VARIANT == "V2" and not GOLD_FILES and not GOLD_FINAL and not RUN_TEST
assert CONFIGS == ["base", "fragmentos"]
assert 0 < VAL_FRAC < 1 and 32 <= CONTEXT_BUDGET < MAX_LENGTH - 3
if RUN_TEST and (not GOLD_FILES or not GOLD_FINAL):
    raise ValueError("La prueba requiere gold final explícito. No usar pre-etiquetas como gold.")


## 2. Verificar GPU, reproducibilidad y versiones

Se usa BF16 en A100, padding dinámico y batches de 16 con dos pasos de acumulación.
El batch efectivo es 32. Cada configuración reinicia SciBERT y la cabeza con la
misma semilla. La reproducibilidad numérica puede variar entre versiones de CUDA
y hardware; se registran el entorno y la revisión exacta del modelo.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import collections
import gc
import hashlib
import importlib.metadata
import json
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support, f1_score)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback, set_seed)
from huggingface_hub import HfApi
from IPython.display import display

set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Activa GPU A100 en Colab antes de entrenar.")
GPU_NAME = torch.cuda.get_device_name(0)
if REQUIRE_A100 and "A100" not in GPU_NAME:
    raise RuntimeError(f"GPU asignada: {GPU_NAME}. Cambia a A100 o ajusta REQUIRE_A100.")
USE_BF16 = torch.cuda.is_bf16_supported()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
if RESUME and {"base", "fragmentos"} <= set(CONFIGS) and not (OUTPUT_ROOT / "base/TRAINING_COMPLETE.json").is_file():
    raise FileNotFoundError("No está la exportación base terminada de este RUN_ID. Revisa Drive antes de continuar con fragmentos.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def freeze_json(path, value):
    path = Path(path)
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != value:
        raise ValueError(f"RUN_ID ya tiene otra configuración o datos: {path.name}. Usa otro RUN_ID.")
    save_json(path, value)

def read_jsonl(path):
    with Path(path).open(encoding="utf-8-sig") as f:
        for n, line in enumerate(f, 1):
            if line.strip():
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"JSON inválido: {path}:{n}") from e
                if not isinstance(row, dict):
                    raise ValueError(f"Se esperaba un objeto: {path}:{n}")
                yield row

def unique_index(rows, name):
    index = {}
    for row in rows:
        key = row["inst_key"]
        if key in index:
            raise ValueError(f"inst_key repetida en {name}: {key}")
        index[key] = row
    return index

ENVIRONMENT = {"python": sys.version, "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": GPU_NAME,
               "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
               "bf16": USE_BF16,
               "packages": {p.split("==")[0]: importlib.metadata.version(p.split("==")[0]) for p in PINNED}}
freeze_json(OUTPUT_ROOT / "request_manifest.json", {
    "variant": VARIANT, "seed": SEED, "split_seed": SPLIT_SEED, "val_frac": VAL_FRAC,
    "max_length": MAX_LENGTH, "context_budget": CONTEXT_BUDGET, "epochs": EPOCHS,
    "lr": LEARNING_RATE, "train_batch": TRAIN_BATCH, "grad_accum": GRAD_ACCUM,
    "eval_batch": EVAL_BATCH, "gold_final": GOLD_FINAL, "run_test": RUN_TEST,
    "ids_dir": str(IDS_DIR), "batch_dir": str(BATCH_DIR), "labels": LABELS,
    "gold_hashes": {str(p): sha256(p) for p in GOLD_FILES},
    "metadata_hashes": {p.name: sha256(p) for p in sorted(IDS_DIR.glob("*.json*"))},
    "environment": ENVIRONMENT,
})
save_json(OUTPUT_ROOT / "environment.json", ENVIRONMENT)
print(json.dumps(ENVIRONMENT, indent=2))


## 3. Verificar los archivos de G15 y sus hashes

El manifiesto declara 31.265 filas V0 y 3.510 paquetes propios de prueba contando
activos, reserva y colchón. **No son 3.510 etiquetas humanas finales.** Todos sus
citantes se excluyen antes del split, aunque algún paquete nunca llegue a anotarse.
Los archivos de calibración eval60/calib30/calib2 no son entrenamiento ni prueba.

Esta celda verifica todos los hashes locales declarados. El pool completo es
opcional: aporta votos, licencias y localizadores, pero no sustituye los lotes con texto.
Si está presente, se comprueba contra su hash de origen. Los cuatro muestreos de A3
no se exigen porque aquí se consume su derivado congelado `test_set_ids_G15`.

También se verifican el manifiesto y la auditoría de exportación FP32 de V0 y se
registran hashes de los reportes/predicciones usados como referencia. No se reescriben.


In [ ]:
manifest_path = IDS_DIR / "manifest_training_v1_G15.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Sube TrainingData_Grupo15 y ajusta IDS_DIR: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["group"] == "G15" and manifest["test_set"] == "B"
assert manifest["variant"] == "V0"
input_hashes = {}
for filename, expected in manifest["files"].items():
    path = IDS_DIR / filename
    actual = sha256(path)
    if actual != expected:
        raise ValueError(f"Hash distinto al manifiesto: {filename}")
    input_hashes[str(path)] = actual
input_hashes[str(manifest_path)] = sha256(manifest_path)
if POOL_PATH.is_file():
    actual = sha256(POOL_PATH)
    if actual != manifest["pool_sha256"]:
        raise ValueError("El pool no corresponde al manifiesto de entrenamiento.")
    input_hashes[str(POOL_PATH)] = actual

train_v0 = list(read_jsonl(IDS_DIR / "training_v1_G15.jsonl"))
test_ids = list(read_jsonl(IDS_DIR / "test_set_ids_G15.jsonl"))
train_index = unique_index(train_v0, "training_v1_G15")
test_index = unique_index(test_ids, "test_set_ids_G15")
assert len(train_v0) == manifest["rows_kept"]
assert len(test_ids) == manifest["test_set_ids"]
assert not (train_index.keys() & test_index.keys())
for row in train_v0 + test_ids:
    citing, para, span, cited = row["inst_key"].split("|")
    assert citing == row["citing_paper_id"] and cited == row["cited_paper_id"]
    assert citing != cited
    assert row["group_key"].startswith(f"{citing}|{para}|")
assert all(r["high_confidence"] and r["primary_pre_label"] in LABEL2ID for r in train_v0)
assert all(r["set"] == "B" for r in test_ids)
print(f"V0: {len(train_v0):,} filas; paquetes propios excluidos: {len(test_ids):,}.")

"""Se incrusta en el notebook V2: referencias V0 y comparación sin consultar T_B."""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def reference_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_v0_reference(root, evaluation_root, labels):
    root, evaluation_root = Path(root), Path(evaluation_root)
    spec = json.loads((root / "run_manifest.json").read_text(encoding="utf-8"))
    if spec["isolation"] != "V0" or spec["labels"] != labels:
        raise ValueError("La referencia debe ser V0 y usar las mismas nueve categorías.")
    required = [root / "run_manifest.json", root / "split_class_counts.csv",
                evaluation_root / "validation_comparison.csv"]
    summary = pd.read_csv(required[-1])
    if set(summary.config) != {"base", "fragmentos"} or len(summary) != 2:
        raise ValueError("Faltan las dos referencias V0 o hay configuraciones duplicadas.")
    if not summary.precision.eq("fp32").all():
        raise ValueError("La comparación debe usar referencias FP32.")
    for config in ["base", "fragmentos"]:
        prediction_path = evaluation_root / config / "validation_predictions.csv"
        audit_path = evaluation_root / config / "export_precision_audit.json"
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if audit["export_check"] != "passed_full_validation":
            raise ValueError(f"La exportación V0 de {config} no pasó la auditoría.")
        if audit["split_hashes"] != spec["split_hashes"]:
            raise ValueError("La evaluación V0 pertenece a otra partición.")
        frame = pd.read_csv(prediction_path)
        validate_prediction_frame(frame, labels)
        if len(frame) != int(summary.loc[summary.config.eq(config), "n"].iloc[0]):
            raise ValueError("El tamaño de las predicciones V0 no coincide con su reporte.")
        required.extend([prediction_path, audit_path])
    provenance = {"run_id": root.name, "evaluation_root": str(evaluation_root),
                  "isolation": "V0", "files_sha256": {str(p): reference_digest(p) for p in required}}
    return spec, provenance


def validate_v2_spec(reference, current):
    if reference["isolation"] != "V0" or current["isolation"] != "V2":
        raise ValueError("Se espera una referencia V0 y una corrida nueva V2.")
    allowed = {"isolation", "split_hashes", "environment"}
    changes = [key for key in sorted(set(reference) | set(current))
               if key not in allowed and reference.get(key) != current.get(key)]
    if changes:
        raise ValueError(f"Además del aislamiento cambiaron datos/hiperparámetros: {changes}")


def validate_prediction_frame(frame, labels):
    required = {"inst_key", "gold_or_prelabel", "label_source", "predicted_label"}
    if not required <= set(frame.columns) or frame.empty:
        raise ValueError("Predicciones vacías o sin claves/etiquetas/procedencia.")
    if frame.inst_key.isna().any() or frame.inst_key.duplicated().any():
        raise ValueError("Claves ausentes o repetidas en las predicciones.")
    if not frame.gold_or_prelabel.isin(labels).all() or not frame.predicted_label.isin(labels).all():
        raise ValueError("Categoría desconocida en las predicciones.")
    if not frame.label_source.eq("pre_label").all():
        raise ValueError("Esta prueba compara únicamente las preetiquetas A2 congeladas.")


def isolation_scores(frame, labels):
    truth, pred = frame.gold_or_prelabel, frame.predicted_label
    metrics = {"n": len(frame), "accuracy": float(accuracy_score(truth, pred))}
    for average in ["macro", "micro"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            truth, pred, labels=labels, average=average, zero_division=0)
        metrics.update({f"precision_{average}": float(precision), f"recall_{average}": float(recall),
                        f"f1_{average}": float(f1)})
    return metrics


def compare_v0_v2(reference_root, current_root, reference_run, current_run, labels,
                  reference_train_n, current_train_n, output_dir):
    frames, full_rows, common_rows, class_rows = {}, [], [], []
    for isolation, root in [("V0", reference_root), ("V2", current_root)]:
        for config in ["base", "fragmentos"]:
            frame = pd.read_csv(Path(root) / config / "validation_predictions.csv")
            validate_prediction_frame(frame, labels)
            frames[isolation, config] = frame.set_index("inst_key", drop=False)
    reference_keys = set(frames["V0", "base"].index)
    current_keys = set(frames["V2", "base"].index)
    if set(frames["V0", "fragmentos"].index) != reference_keys:
        raise ValueError("Base y Top-3 de V0 no usan la misma validación.")
    if set(frames["V2", "fragmentos"].index) != current_keys:
        raise ValueError("Base y Top-3 de V2 no usan la misma validación.")
    # Mismos archivos y hash por citante: la validación V2 debe ser subconjunto de V0.
    if not current_keys <= reference_keys:
        raise ValueError("V2 contiene claves de validación ajenas a la referencia V0.")
    keys = sorted(current_keys)
    truth = frames["V0", "base"].loc[keys, "gold_or_prelabel"]
    for (isolation, config), frame in frames.items():
        aligned = frame.loc[keys]
        if not np.array_equal(aligned.gold_or_prelabel.to_numpy(), truth.to_numpy()):
            raise ValueError("Las etiquetas no coinciden al alinear por inst_key.")
        identity = {"run_id": reference_run if isolation == "V0" else current_run,
                    "isolation": isolation, "config": config, "precision": "fp32",
                    "train_n": int(reference_train_n if isolation == "V0" else current_train_n)}
        full_rows.append({**identity, "scope": "own_validation", **isolation_scores(frame, labels)})
        common_rows.append({**identity, "scope": "common_validation", **isolation_scores(aligned, labels)})
        precision, recall, f1, support = precision_recall_fscore_support(
            aligned.gold_or_prelabel, aligned.predicted_label, labels=labels, zero_division=0)
        class_rows.extend({**identity, "label": label, "precision_class": float(p),
                           "recall_class": float(r), "f1_class": float(f), "support": int(n)}
                          for label, p, r, f, n in zip(labels, precision, recall, f1, support))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    full, common, per_class = map(pd.DataFrame, [full_rows, common_rows, class_rows])
    full.to_csv(output_dir / "isolation_own_validation.csv", index=False)
    common.to_csv(output_dir / "isolation_common_validation.csv", index=False)
    per_class.to_csv(output_dir / "isolation_common_per_class.csv", index=False)
    pd.DataFrame({"inst_key": keys}).to_csv(output_dir / "isolation_common_keys.csv", index=False)
    return full, common, per_class

BASELINE_SPEC, V0_REFERENCE = load_v0_reference(BASELINE_ROOT, BASELINE_EVALUATION_ROOT, LABELS)
save_json(OUTPUT_ROOT / "v0_reference.json", V0_REFERENCE)
print("Referencia V0 verificada:", V0_REFERENCE["evaluation_root"])


## 4. EDA breve: distribución de clases e impacto del aislamiento

La gráfica muestra las **preetiquetas de A2** de G15 antes de separar train/val:
una torta de la variante seleccionada y barras con los conteos de V0 como referencia.
El total es el conjunto disponible después de las exclusiones, **no el pool completo
ni el test humano**. V0/V1/V2 indican aislamiento; el archivo de origen sigue siendo
`training_v1_G15.jsonl`. Dos líneas resumen el tamaño y el desbalance observado.
La figura y los conteos se conservan en los resultados para documentar cada corrida.

Se interpretan literalmente las listas por `role`: `citing` comprueba el citante y
`cited` el citado. V2 ya contiene las reglas para ambos papeles. No se simplifica a
una lista genérica que cambie el significado de V0 o V1.

Se registra la pérdida por clase antes de elegir. V0 permite más datos y requiere
reportar después el rendimiento sobre citados vistos/no vistos. V1/V2 sacrifican
datos, especialmente de las clases escasas. La comparación de configuraciones
usa siempre la misma variante y las mismas instancias.


In [ ]:
def exclusion_rules(rows):
    rules = {"citing": set(), "cited": set()}
    for row in rows:
        if row["group"] != "G15" or row["role"] not in rules:
            raise ValueError(f"Regla de exclusión inesperada: {row}")
        rules[row["role"]].add(row["paper_id"])
    return rules

def excluded(row, rules):
    return (row["citing_paper_id"] in rules["citing"] or
            row["cited_paper_id"] in rules["cited"])

RULES = {v: exclusion_rules(read_jsonl(IDS_DIR / f"exclusion_list_G15_{v}.jsonl"))
         for v in ("V0", "V1", "V2")}
assert not any(excluded(r, RULES["V0"]) for r in train_v0)
variant_rows = {v: [r for r in train_v0 if not excluded(r, rules)] for v, rules in RULES.items()}
impact = pd.DataFrame({v: collections.Counter(r["primary_pre_label"] for r in rows)
                       for v, rows in variant_rows.items()}).reindex(LABELS).fillna(0).astype(int)
impact.loc["TOTAL"] = impact.sum()
impact.to_csv(OUTPUT_ROOT / "isolation_impact.csv")
display(impact)
eligible = [dict(r) for r in variant_rows[VARIANT]]
assert not ({r["citing_paper_id"] for r in eligible} & {r["citing_paper_id"] for r in test_ids})
save_json(OUTPUT_ROOT / "isolation_summary.json", {
    "variant": VARIANT, "pool_high_conf_rows": manifest["high_conf_rows"],
    "removed_to_v0": manifest["rows_removed"], "v0_rows": len(train_v0),
    "removed_beyond_v0": len(train_v0) - len(eligible), "eligible_rows": len(eligible),
})

import matplotlib.pyplot as plt

eda_counts = impact.loc[LABELS, ["V0"] if VARIANT == "V0" else ["V0", VARIANT]].copy()
eda_counts.index.name = "label"
eda_root = OUTPUT_ROOT  # los reportes de MLflow incluyen PNG/CSV de esta raíz
eda_root.mkdir(parents=True, exist_ok=True)
eda_counts.to_csv(eda_root / "eda_class_distribution.csv")
selected_counts = eda_counts[VARIANT]
eda_total = int(selected_counts.sum())
colors = plt.get_cmap("tab10")(np.arange(len(LABELS)))
fig, (pie_ax, bar_ax) = plt.subplots(1, 2, figsize=(17, 6))
wedges, _, _ = pie_ax.pie(selected_counts, colors=colors, startangle=90,
    autopct=lambda pct: f"{pct:.1f}%" if pct >= 3 else "",
    pctdistance=0.78, wedgeprops={"width": 0.42, "edgecolor": "white"})
pie_ax.text(0, 0, f"{eda_total:,}\nejemplos", ha="center", va="center", fontsize=15)
pie_ax.set_title(f"Distribución de clases · {VARIANT}")
pie_ax.legend(wedges, [f"{label}: {int(n):,} ({n / eda_total:.1%})"
    for label, n in selected_counts.items()], loc="upper center",
    bbox_to_anchor=(0.5, -0.02), fontsize=8, ncol=2)
eda_counts.plot.barh(ax=bar_ax, color=["#8fa6be", "#16877c"][:len(eda_counts.columns)])
bar_ax.invert_yaxis()
bar_ax.set(xlabel="Número de ejemplos", ylabel="", title="Conteos antes del split train/val")
for container in bar_ax.containers:
    bar_ax.bar_label(container, fmt="%.0f", padding=3, fontsize=8)
bar_ax.set_xlim(0, int(eda_counts.max().max()) * 1.22)
bar_ax.grid(axis="x", alpha=0.2)
fig.suptitle(f"EDA G15 · origen training v1 · aislamiento {VARIANT} · preetiquetas A2")
fig.tight_layout()
fig.savefig(eda_root / "eda_class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
eda_majority = selected_counts.idxmax()
eda_minority = selected_counts.idxmin()
print(f"1. {VARIANT}: {eda_total:,} ejemplos en {(selected_counts > 0).sum()} clases; "
      f"se conserva {eda_total / len(train_v0):.1%} de V0 antes de separar train/val.")
eda_min_count = int(selected_counts.min())
eda_ratio = f"{selected_counts.max() / eda_min_count:.1f} veces" if eda_min_count else "no calculable (clase vacía)"
print(f"2. Mayoritaria: {eda_majority} ({selected_counts.max() / eda_total:.1%}); "
      f"minoritaria: {eda_minority} ({eda_min_count:,}); razón mayor/menor: {eda_ratio}. "
      "Son preetiquetas, no resultados contra gold humano.")


## 5. Incorporar gold humano disponible sin usar T_B para entrenamiento

Se acepta `gold_primary` unido exclusivamente por `inst_key`. Si no hay archivos
gold, entrenamiento y validación usan las pre-etiquetas de A2, incluido T_A.
Cuando hay gold final de T_A se reemplaza su pre-etiqueta; `unclear` se elimina.
Las secundarias no se usan para esta clasificación de etiqueta única.

Los archivos gold pueden contener ambos conjuntos: las claves de T_B jamás se
añaden a train/val. Un `gold_primary=null` significa aún sin resolver y mantiene
la pre-etiqueta en train/val. Un archivo con correcciones conflictivas requiere
consolidación explícita; no se resuelve por orden de lectura. Cambiar gold exige
otro `RUN_ID` y reentrenar las dos configuraciones para compararlas.


In [ ]:
gold = {}
for filename in GOLD_FILES:
    path = Path(filename)
    input_hashes[str(path)] = sha256(path)
    for row in read_jsonl(path):
        key = row["inst_key"]
        label = row.get("gold_primary")
        if label is None:
            continue
        if label not in LABEL2ID and label != "unclear":
            raise ValueError(f"Gold desconocido: {key}: {label}")
        if key in gold and gold[key] != label:
            raise ValueError(f"Gold conflictivo; consolida versiones: {key}")
        gold[key] = label

removed_unclear = []
labeled = []
for row in eligible:
    key = row["inst_key"]
    label = gold.get(key, row["primary_pre_label"])
    if label == "unclear":
        removed_unclear.append(key)
        continue
    row["label"] = label
    row["label_source"] = "human_gold" if key in gold else "pre_label"
    labeled.append(row)
test_rows = []
if RUN_TEST:
    for row in test_ids:
        label = gold.get(row["inst_key"])
        if label in LABEL2ID:
            test_rows.append({**row, "label": label, "label_source": "human_gold"})
    counts = collections.Counter(r["label"] for r in test_rows)
    if any(counts[label] < 300 for label in LABELS):
        raise ValueError(f"T_B no alcanza 300 gold válidos por clase: {dict(counts)}. Cierra A3.")
save_json(OUTPUT_ROOT / "label_provenance.json", {
    "train_val_sources": dict(collections.Counter(r["label_source"] for r in labeled)),
    "removed_unclear_keys": removed_unclear, "final_test_enabled": RUN_TEST,
    "human_test_rows": len(test_rows),
})
print("Fuentes train/val:", collections.Counter(r["label_source"] for r in labeled))
print("Test humano habilitado:", RUN_TEST, "filas:", len(test_rows))


## 6. Crear una sola partición train/val y auditarla

El hash del citante, con semilla fija, asigna aproximadamente el 10% a validación.
La proporción exacta por clase puede variar; se exige presencia de las nueve clases
en ambas particiones y se publican sus conteos. Esta asignación no depende de los
textos ni de predicciones. El grupo de citas y el par citante–citado quedan juntos.

En V1 se purga además de train todo citado presente en val; en V2 se purgan ambos
roles de cualquier documento de val. La val no se modifica para favorecer resultados.
Si el suministro resulta insuficiente, se revisa el diseño antes de entrenar.
Los archivos de partición exportados contienen claves, etiquetas y procedencia, sin texto.


In [ ]:
def validation_citing(citing_id):
    digest = hashlib.sha256(f"{SPLIT_SEED}|val|{citing_id}".encode()).hexdigest()
    return int(digest[:16], 16) / 2**64 < VAL_FRAC

val_rows = [r for r in labeled if validation_citing(r["citing_paper_id"])]
tr_before_purge = [r for r in labeled if not validation_citing(r["citing_paper_id"])]
val_citing = {r["citing_paper_id"] for r in val_rows}
val_cited = {r["cited_paper_id"] for r in val_rows}
if VARIANT == "V0":
    train_rows = tr_before_purge
elif VARIANT == "V1":
    train_rows = [r for r in tr_before_purge if r["cited_paper_id"] not in val_cited]
else:
    val_docs = val_citing | val_cited
    train_rows = [r for r in tr_before_purge if r["citing_paper_id"] not in val_docs
                  and r["cited_paper_id"] not in val_docs]
assert train_rows and val_rows
splits = {"train": train_rows, "validation": val_rows}
if RUN_TEST:
    splits["test"] = test_rows
for a, ar in splits.items():
    assert all(r["label"] in LABEL2ID for r in ar)
    if a != "test":
        assert not any(excluded(r, RULES[VARIANT]) for r in ar)
    for b, br in splits.items():
        if a >= b:
            continue
        for key in ("inst_key", "group_key", "citing_paper_id"):
            assert not ({r[key] for r in ar} & {r[key] for r in br}), (a, b, key)
        if VARIANT == "V1":
            assert not ({r["cited_paper_id"] for r in ar} & {r["cited_paper_id"] for r in br})
        if VARIANT == "V2":
            da = {r[k] for r in ar for k in ("citing_paper_id", "cited_paper_id")}
            db = {r[k] for r in br for k in ("citing_paper_id", "cited_paper_id")}
            assert not da & db
counts = pd.DataFrame({s: collections.Counter(r["label"] for r in rows)
                       for s, rows in splits.items()}).reindex(LABELS).fillna(0).astype(int)
if (counts[["train", "validation"]] == 0).any().any():
    raise ValueError("Alguna clase está ausente en train o val. Revisa el diseño del split.")
display(counts)
counts.to_csv(OUTPUT_ROOT / "split_class_counts.csv")
KEY_FIELDS = ["inst_key", "group_key", "citing_paper_id", "cited_paper_id", "label", "label_source", "source_batch"]
split_hashes = {}
for name, rows in splits.items():
    rows.sort(key=lambda r: r["inst_key"])
    path = OUTPUT_ROOT / f"split_{name}_ids.jsonl"
    content = "".join(json.dumps({k: r[k] for k in KEY_FIELDS}, ensure_ascii=False, sort_keys=True) + "\n" for r in rows)
    split_hashes[name] = hashlib.sha256(content.encode()).hexdigest()
    path.write_text(content, encoding="utf-8")
save_json(OUTPUT_ROOT / "split_summary.json", {
    "split_seed": SPLIT_SEED, "validation_fraction_target": VAL_FRAC,
    "removed_for_validation_isolation": len(tr_before_purge) - len(train_rows),
    "rows": {k: len(v) for k, v in splits.items()}, "hashes": split_hashes,
    "temporal_split": False, "section_stratification": False,
})


## 7. Localizar los lotes y revisar su esquema real

Se necesitan estos ocho archivos en `a2/batches_norw/`:
`pilot_random_1500`, `pool_single_7000`, `pool_cue_eg`, `pool_cue_cmp`,
`pool_cue_rare`, `pool_cue_yield`, `pool_cue_basis`, `pool_ext_cue_rare` (todos `.jsonl`).

El esquema se verificó con los ocho lotes locales de `A4/a2/batches_norw/`:
el contexto está en `context` y los fragmentos en `top3`. Esta celda vuelve a mostrar
claves y tipos sin imprimir textos. Si se utiliza una nueva versión con otro esquema,
editar `BATCH_FIELDS`/`CHUNK_FIELDS` con los nombres reales antes de continuar.
Para campos anidados usar puntos, p. ej. `cited.title`.
No se inventan textos ni se cruzan filas por orden o por parecido.

Si los lotes no llevan `inst_key`, se necesita el adaptador original de A2 que la
construya a partir de sus localizadores; el notebook se detiene para evitar uniones falsas.


In [ ]:
BATCH_FIELDS = {
    "inst_key": "inst_key", "context": "context",
    "title": "cited_title", "abstract": "cited_abstract", "chunks": "top3",
}
CHUNK_FIELDS = {"text": "text", "section": "section"}

def field_at(row, dotted):
    value = row
    for part in dotted.split("."):
        if not isinstance(value, dict) or part not in value:
            raise KeyError(f"Campo '{dotted}' ausente. Ajusta el adaptador con el esquema real.")
        value = value[part]
    return value

required_batches = sorted({r["source_batch"] for rows in splits.values() for r in rows})
batch_paths = {name: BATCH_DIR / f"{name}.jsonl" for name in required_batches}
missing = [str(p) for p in batch_paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Faltan lotes con texto; descárgalos de a2/batches_norw/:\n" + "\n".join(missing))
for name, path in batch_paths.items():
    first = next(read_jsonl(path), None)
    if first is None:
        raise ValueError(f"Lote vacío: {name}")
    print(name, {k: type(v).__name__ for k, v in first.items()})
    chunks = field_at(first, BATCH_FIELDS["chunks"])
    if isinstance(chunks, list) and chunks:
        print("  primer chunk:", {k: type(v).__name__ for k, v in chunks[0].items()})
freeze_json(OUTPUT_ROOT / "batch_manifest.json", {
    "hashes": {name: sha256(path) for name, path in batch_paths.items()},
    "batch_fields": BATCH_FIELDS, "chunk_fields": CHUNK_FIELDS,
})
print("Si el esquema coincide, continúa. Si difiere, corrige el adaptador y repite esta celda.")


## 8. Reconstruir texto por clave y controlar calidad

Se leen secuencialmente solo los ocho lotes y se conservan en memoria las instancias
requeridas. Cada fila debe tener un único `[[TARGET]]`, título, resumen y exactamente
tres fragmentos en el orden de recuperación. Los lotes reales contienen algunas
secciones vacías: se representan como `Unknown` y se cuentan en la auditoría, sin
inventar una sección retórica ni eliminar la instancia. Un texto faltante o una
clave duplicada detiene la ejecución. No se descartan silenciosamente filas distintas
en cada configuración. La etiqueta, los votos y los identificadores no forman parte
del texto de entrada del modelo.

Los datos reconstruidos permanecen en memoria; no se exporta el corpus textual.


In [ ]:
def require_text(value, name):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Texto vacío o tipo incorrecto: {name}")
    return value.strip()

def normalize_batch(row):
    out = {k: require_text(field_at(row, BATCH_FIELDS[k]), k)
           for k in ("inst_key", "context", "title", "abstract")}
    if out["context"].count("[[TARGET]]") != 1:
        raise ValueError(f"Se requiere un [[TARGET]] por contexto: {out['inst_key']}")
    if "{{cite:" in out["context"]:
        raise ValueError("Quedan marcadores crudos. Usa el renderizador congelado de A2.")
    raw_chunks = field_at(row, BATCH_FIELDS["chunks"])
    if not isinstance(raw_chunks, list) or len(raw_chunks) != 3:
        raise ValueError(f"Top-3 incompleto: {out['inst_key']}")
    out["chunks"] = []
    for ch in raw_chunks:
        text = require_text(field_at(ch, CHUNK_FIELDS["text"]), "chunk.text")
        section = field_at(ch, CHUNK_FIELDS["section"])
        if section is not None and not isinstance(section, str):
            raise ValueError(f"Tipo incorrecto de sección: {out['inst_key']}")
        missing_section = section is None or not section.strip()
        out["chunks"].append({"text": text, "section": "Unknown" if missing_section else section.strip(),
                              "section_missing": missing_section})
    return out

needed = {r["inst_key"]: r["source_batch"] for rows in splits.values() for r in rows}
texts = {}
for batch, path in batch_paths.items():
    input_hashes[str(path)] = sha256(path)
    seen_batch = set()
    for raw in read_jsonl(path):
        key = field_at(raw, BATCH_FIELDS["inst_key"])
        if key in seen_batch:
            raise ValueError(f"Clave duplicada en {batch}: {key}")
        seen_batch.add(key)
        if key in needed and needed[key] == batch:
            texts[key] = normalize_batch(raw)
missing_keys = sorted(needed.keys() - texts.keys())
if missing_keys:
    raise ValueError(f"Sin reconstruir: {len(missing_keys)}. Primeras claves: {missing_keys[:5]}")
assert len(texts) == len(needed)
save_json(OUTPUT_ROOT / "reconstruction_audit.json", {
    "required_rows": len(needed), "reconstructed_rows": len(texts),
    "batches": required_batches, "adapter": BATCH_FIELDS, "chunk_adapter": CHUNK_FIELDS,
    "rows_with_missing_section": sum(any(ch["section_missing"] for ch in ex["chunks"]) for ex in texts.values()),
    "chunks_with_missing_section": sum(ch["section_missing"] for ex in texts.values() for ch in ex["chunks"]),
})
section_rows = []
for split, rows in splits.items():
    cnt = collections.Counter(ch["section"] for r in rows for ch in texts[r["inst_key"]]["chunks"])
    section_rows.extend({"split": split, "section": sec, "chunks": n} for sec, n in cnt.items())
pd.DataFrame(section_rows).to_csv(OUTPUT_ROOT / "section_distribution.csv", index=False)
print(f"Reconstruidas {len(texts):,} instancias con ambas configuraciones completas.")


## 9. Resolver el límite de 512 tokens sin perder la cita objetivo

SciBERT admite 512 tokens incluyendo `[CLS]` y los dos `[SEP]` del par. Se reserva
una ventana de hasta **192 tokens del contexto**, centrada en `[[TARGET]]`; es idéntica
en las dos configuraciones. El resto del presupuesto se usa para el citado:

- Base: título (hasta 64 tokens) y abstract con el espacio restante.
- Fragmentos: presupuesto repartido entre los tres, con sección (hasta 24 tokens)
  y texto de cada uno; el espacio sobrante se redistribuye.

Es una decisión de A4: A1 evitaba truncar contextos, pero concatenar toda la evidencia
supera el límite. Aquí se registra cuánto se recorta y siempre se conserva el marcador.
Se añaden cinco marcadores al vocabulario, cuyas representaciones se aprenden en el
ajuste. La política y el tokenizador se exportan juntos para inferencia idéntica.


In [ ]:
MODEL_ID = "allenai/scibert_scivocab_uncased"
previous_spec = BASELINE_ROOT / "run_manifest.json"
MODEL_REVISION = (json.loads(previous_spec.read_text(encoding="utf-8"))["model_revision"]
                  if previous_spec.exists() else HfApi().model_info(MODEL_ID).sha)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
MARKERS = ["[[TARGET]]", "[CITE]", "[FORMULA]", "[FIGURE]", "[TABLE]"]
tokenizer.add_special_tokens({"additional_special_tokens": MARKERS})
TARGET_ID = tokenizer.convert_tokens_to_ids("[[TARGET]]")

def fair_caps(lengths, budget):
    caps = [0] * len(lengths)
    remaining = budget
    while remaining and any(c < n for c, n in zip(caps, lengths)):
        for i, n in enumerate(lengths):
            if remaining and caps[i] < n:
                caps[i] += 1
                remaining -= 1
    return caps

def encode_example(example, config, tok, max_length=512, context_budget=192):
    ids = lambda s: tok.encode(s, add_special_tokens=False)
    context = ids(example["context"])
    target_id = tok.convert_tokens_to_ids("[[TARGET]]")
    if context.count(target_id) != 1:
        raise ValueError("El contexto debe contener exactamente un marcador objetivo.")
    target = context.index(target_id)
    start = max(0, min(target - context_budget // 2, len(context) - context_budget))
    context_cut = context[start:start + context_budget]
    assert tok.num_special_tokens_to_add(pair=True) == 3, "Se requiere tokenizador BERT."
    budget = max_length - 3 - len(context_cut)
    assert budget >= 32 and target_id in context_cut
    stats = {"context_original": len(context), "context_kept": len(context_cut)}
    if config == "base":
        title = ids("Title: " + example["title"])
        abstract = ids("Abstract: " + example["abstract"])
        title_kept = title[:min(64, budget // 3)]
        abstract_kept = abstract[:budget - len(title_kept)]
        evidence = title_kept + abstract_kept
        stats.update(evidence_original=len(title) + len(abstract), evidence_kept=len(evidence),
                     title_original=len(title), title_kept=len(title_kept),
                     abstract_original=len(abstract), abstract_kept=len(abstract_kept))
    elif config == "fragmentos":
        sections = [ids(f"Section: {ch['section']} Passage: ")[:24] for ch in example["chunks"]]
        bodies = [ids(ch["text"]) for ch in example["chunks"]]
        caps = fair_caps([len(h) + len(b) for h, b in zip(sections, bodies)], budget)
        evidence = []
        for i, (head, body, cap) in enumerate(zip(sections, bodies, caps), 1):
            kept = (head + body)[:cap]
            if len(kept) <= len(head):
                raise ValueError("Presupuesto insuficiente para conservar texto de cada chunk.")
            evidence.extend(kept)
            stats[f"chunk{i}_original"] = len(head) + len(body)
            stats[f"chunk{i}_kept"] = len(kept)
        stats.update(evidence_original=sum(len(h) + len(b) for h, b in zip(sections, bodies)),
                     evidence_kept=len(evidence))
    else:
        raise ValueError(config)
    input_ids = [tok.cls_token_id] + context_cut + [tok.sep_token_id] + evidence + [tok.sep_token_id]
    token_types = [0] * (len(context_cut) + 2) + [1] * (len(evidence) + 1)
    assert len(input_ids) <= max_length and len(token_types) == len(input_ids)
    stats["total_tokens"] = len(input_ids)
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "token_type_ids": token_types}, stats

encoded = {}
audit_rows = []
for config in CONFIGS:
    encoded[config] = {}
    for split, rows in splits.items():
        features = []
        for row in rows:
            features_row, stats = encode_example(texts[row["inst_key"]], config, tokenizer,
                                                  MAX_LENGTH, CONTEXT_BUDGET)
            features.append({**features_row, "labels": LABEL2ID[row["label"]]})
            audit_rows.append({"config": config, "split": split, "inst_key": row["inst_key"], **stats})
        encoded[config][split] = Dataset.from_list(features)
audit = pd.DataFrame(audit_rows)
audit["context_truncated"] = audit.context_kept < audit.context_original
audit["evidence_truncated"] = audit.evidence_kept < audit.evidence_original
audit.to_csv(OUTPUT_ROOT / "token_budget_audit.csv", index=False)
display(audit.groupby(["config", "split"])[["total_tokens", "context_truncated", "evidence_truncated"]].mean())


## 10. Congelar el experimento y medir una referencia simple

Se guarda la revisión de SciBERT, versiones, hashes de datos y particiones, adaptador
y parámetros. Una ejecución existente con otra configuración se rechaza: cambiar
`RUN_ID`. Este manifiesto permite retomar entrenamiento sin mezclar datasets.

La referencia mayoritaria se aprende en train y se evalúa en val. Val contiene
principalmente etiquetas de jueces: sus métricas sirven para seleccionar checkpoints,
no equivalen a desempeño humano sobre T_B. El test no participa en selección de
épocas, variantes, learning rate ni política de tokens.

Se exige igualdad de datos, etiquetas, revisión del modelo e hiperparámetros con V0;
solo cambian aislamiento, hashes de partición y entorno registrado. Una nueva entrega
de datos requerirá otro RUN_ID y una comparación diseñada para esa entrega.


In [ ]:
run_spec = {
    "group": "G15", "test_set": "B", "isolation": VARIANT,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "seed": SEED,
    "split_seed": SPLIT_SEED, "split_hashes": split_hashes, "input_hashes": input_hashes,
    "labels": LABELS, "markers": MARKERS, "max_length": MAX_LENGTH,
    "context_budget": CONTEXT_BUDGET, "title_budget": 64, "section_budget": 24,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "train_batch": TRAIN_BATCH,
    "gradient_accumulation": GRAD_ACCUM, "eval_batch": EVAL_BATCH,
    "weight_decay": 0.01, "warmup_ratio": 0.1, "early_stopping_patience": 2,
    "loss": "unweighted_cross_entropy", "gold_final": GOLD_FINAL, "run_test": RUN_TEST,
    "batch_fields": BATCH_FIELDS, "chunk_fields": CHUNK_FIELDS, "environment": ENVIRONMENT,
}
validate_v2_spec(BASELINE_SPEC, run_spec)
run_spec.update(baseline_run_id=BASELINE_ROOT.name, baseline_reference=V0_REFERENCE,
                purpose="exploratory_isolation_V2", dataset_release="training_v1")
spec_path = OUTPUT_ROOT / "run_manifest.json"
if spec_path.exists():
    old_spec = json.loads(spec_path.read_text(encoding="utf-8"))
    if old_spec != run_spec:
        raise ValueError("RUN_ID ya existe con otro manifiesto. Usa otro RUN_ID.")
save_json(spec_path, run_spec)
majority = collections.Counter(r["label"] for r in train_rows).most_common(1)[0][0]
y_val = [LABEL2ID[r["label"]] for r in val_rows]
baseline_metrics = {"class": majority,
    "validation_macro_f1": f1_score(y_val, [LABEL2ID[majority]] * len(y_val),
                                     labels=list(ID2LABEL), average="macro", zero_division=0)}
save_json(OUTPUT_ROOT / "majority_baseline.json", baseline_metrics)
print(baseline_metrics)


## 10.1. Activar MLflow y recuperar el historial de Drive

Se restaura el último snapshot completo antes de abrir la base SQLite local.
Se conserva MLflow 3.17.2, compatible con el visor instalado en AWS.
**Ejecutar una sola sesión escritora de Colab a la vez**, incluido el notebook de instalación.
El experimento real se llama `G15_A4_SciBERT`; cada configuración tiene un padre y
cada intento de entrenamiento un hijo. El experimento de instalación queda separado.

Se publican snapshots verificados al iniciar, cada 300 segundos comprobados al
terminar un paso de optimización, en cada checkpoint y al cerrar cada intento.
La copia pausa brevemente el entrenamiento; no es un temporizador independiente.
Durante preparación/evaluaciones largas no se promete una copia cada cinco minutos.
Una desconexión brusca puede perder métricas posteriores al último snapshot;
la recuperación del entrenamiento parte del último checkpoint de época completo.
Los fallos de respaldo detienen el entrenamiento para evitar continuar sin protección.

Pesos, optimizador y ZIP de modelos quedan en Drive; MLflow guarda reportes,
manifiestos, curvas y referencias. Los snapshots no duplican los pesos.
`report_to="none"` desactiva integraciones automáticas: el callback propio registra
en MLflow los intentos sin duplicar runs ni cargar los modelos como artefactos.


In [ ]:
"""Se incrusta en el notebook: no requiere subir este archivo a Colab."""
import contextlib
import datetime as dt
import math
import numbers
import sqlite3
import tempfile
import uuid
import zipfile
from pathlib import PurePosixPath
import mlflow
from mlflow import MlflowClient
from transformers import TrainerCallback


def tracking_restore(root, snapshots):
    """Solo restaura antes de abrir MLflow; nunca sobrescribe una DB local."""
    root.mkdir(parents=True, exist_ok=True)
    if (root / "mlflow.db").exists():
        return
    candidates = sorted((p.parent for p in snapshots.glob("*/COMPLETE.json")),
        key=lambda p: dt.datetime.fromisoformat(json.loads((p / "COMPLETE.json").read_text())["created_at_utc"]))
    if not candidates:
        print("Sin historial previo: se creará una base MLflow nueva.")
        return
    source = candidates[-1]
    receipt = json.loads((source / "COMPLETE.json").read_text())
    archive_path = source / "snapshot.zip"
    assert receipt["schema_version"] == 1
    assert receipt["snapshot_id"] == source.name
    assert receipt["mlflow_version"] == mlflow.__version__
    assert archive_path.stat().st_size == receipt["zip_size"]
    assert sha256(archive_path) == receipt["zip_sha256"]
    assert (source / "snapshot.sha256").read_text().split()[0] == receipt["zip_sha256"]
    # Validar todo antes de copiar a las rutas canónicas.
    with tempfile.TemporaryDirectory() as tmp:
        target = Path(tmp)
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            assert len(names) == len(set(names))
            manifest = json.loads(archive.read("snapshot_manifest.json"))
            assert manifest["schema_version"] == 1
            assert manifest["snapshot_id"] == receipt["snapshot_id"]
            assert manifest["canonical_root"] == str(root)
            assert manifest["mlflow_version"] == mlflow.__version__
            assert set(names) == set(manifest["files"]) | {"snapshot_manifest.json"}
            for entry in archive.infolist():
                path = PurePosixPath(entry.filename)
                assert not path.is_absolute() and ".." not in path.parts
                assert "\\" not in entry.filename and ":" not in entry.filename
                assert (entry.external_attr >> 16) & 0o170000 != 0o120000
                assert entry.filename in {"mlflow.db", "snapshot_manifest.json"} or entry.filename.startswith("artifacts/")
            archive.extractall(target)
        for name, metadata in manifest["files"].items():
            path = target / name
            assert path.stat().st_size == metadata["size"] and sha256(path) == metadata["sha256"]
        with contextlib.closing(sqlite3.connect(target / "mlflow.db")) as db:
            assert db.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
        assert not (root / "artifacts").exists(), "Hay artefactos locales sin DB; usa una sesión Colab limpia."
        if (target / "artifacts").exists():
            shutil.copytree(target / "artifacts", root / "artifacts")
        shutil.copyfile(target / "mlflow.db", root / "mlflow.db")
    print("Historial restaurado:", source.name)


class A4Tracking:
    def __init__(self, root, snapshots, output, run_id, interval=300):
        self.root, self.snapshots, self.output = Path(root), Path(snapshots), Path(output)
        self.run_id, self.interval = run_id, interval
        self.last_backup = 0
        self.parents, self.attempts = {}, {}
        self.snapshots.mkdir(parents=True, exist_ok=True)
        tracking_restore(self.root, self.snapshots)
        mlflow.set_tracking_uri("sqlite:///" + (self.root / "mlflow.db").as_posix())
        self.client = MlflowClient()
        name = "G15_A4_SciBERT"
        experiment = self.client.get_experiment_by_name(name)
        self.experiment = (experiment.experiment_id if experiment else
            self.client.create_experiment(name, artifact_location=(self.root / "artifacts/scibert").as_uri()))
        # Solo el mismo RUN_ID: las sesiones anteriores terminaron sin cierre.
        previous = self.client.search_runs([self.experiment], filter_string=f"tags.a4_run_id = '{run_id}'")
        for run in previous:
            if run.info.status == "RUNNING":
                self.client.set_terminated(run.info.run_id, status="KILLED")
        self.backup(force=True)

    def backup(self, force=False):
        if not force and time.monotonic() - self.last_backup < self.interval:
            return
        created = dt.datetime.now(dt.timezone.utc)
        sid = created.strftime("%Y%m%dT%H%M%S%fZ_") + uuid.uuid4().hex[:8]
        with tempfile.TemporaryDirectory() as tmp:
            stage = Path(tmp) / "stage"
            stage.mkdir()
            with contextlib.closing(sqlite3.connect(self.root / "mlflow.db")) as source:
                with contextlib.closing(sqlite3.connect(stage / "mlflow.db")) as dest:
                    source.backup(dest)
                    assert dest.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
            if (self.root / "artifacts").exists():
                shutil.copytree(self.root / "artifacts", stage / "artifacts")
            paths = sorted(p for p in stage.rglob("*") if p.is_file())
            assert not any(p.is_symlink() for p in stage.rglob("*"))
            manifest = dict(schema_version=1, snapshot_id=sid, created_at_utc=created.isoformat(),
                mlflow_version=mlflow.__version__, canonical_root=str(self.root), purpose="scibert_training",
                files={p.relative_to(stage).as_posix(): {"sha256": sha256(p), "size": p.stat().st_size} for p in paths})
            save_json(stage / "snapshot_manifest.json", manifest)
            packed = Path(tmp) / "snapshot.zip"
            with zipfile.ZipFile(packed, "w", zipfile.ZIP_DEFLATED) as archive:
                for path in sorted(p for p in stage.rglob("*") if p.is_file()):
                    archive.write(path, path.relative_to(stage).as_posix())
            digest = sha256(packed)
            destination = self.snapshots / sid
            destination.mkdir(exist_ok=False)
            shutil.copyfile(packed, destination / "snapshot.zip")
            assert sha256(destination / "snapshot.zip") == digest
            (destination / "snapshot.sha256").write_text(f"{digest}  snapshot.zip\n")
            save_json(destination / "COMPLETE.json", dict(schema_version=1, snapshot_id=sid,
                created_at_utc=created.isoformat(), mlflow_version=mlflow.__version__,
                zip_sha256=digest, zip_size=packed.stat().st_size))
        self.last_backup = time.monotonic()
        print("MLflow respaldado:", destination)

    def parent(self, config):
        if config not in self.parents:
            runs = self.client.search_runs([self.experiment],
                filter_string=f"tags.a4_run_id = '{self.run_id}' and tags.config = '{config}' and tags.kind = 'configuration'")
            if len(runs) > 1:
                raise ValueError("Más de un padre para la misma configuración")
            self.parents[config] = (runs[0].info.run_id if runs else self.client.create_run(self.experiment,
                tags={"mlflow.runName": f"{self.run_id}/{config}", "a4_run_id": self.run_id,
                      "config": config, "kind": "configuration"}).info.run_id)
        return self.parents[config]

    def metrics(self, run, values, step=0):
        for key, value in values.items():
            if isinstance(value, numbers.Real) and math.isfinite(float(value)):
                self.client.log_metric(run, key, float(value), step=int(step))

    def reports(self, config, run=None):
        run = run or self.parent(config)
        for directory, artifact_path in [(self.output, "provenance"), (self.output / config, "reports")]:
            for path in sorted(directory.iterdir()):
                if path.is_file() and path.suffix in {".json", ".csv", ".png", ".txt"}:
                    self.client.log_artifact(run, str(path), artifact_path=artifact_path)

    @contextlib.contextmanager
    def evaluation(self, config, directory, policy="v2_fp32"):
        parent = self.parent(config)
        rid = self.client.create_run(self.experiment, tags={
            "mlflow.runName": f"{config}/evaluation_{policy}_{uuid.uuid4().hex[:8]}",
            "mlflow.parentRunId": parent, "a4_run_id": self.run_id, "config": config,
            "kind": "evaluation", "evaluation_policy": policy,
            "canonical_precision": "fp32", "drive_output": str(directory)}).info.run_id
        try:
            yield rid
            self.client.log_artifacts(rid, str(directory), artifact_path="evaluation")
            self.client.set_terminated(rid, "FINISHED")
            self.client.set_tag(parent, "latest_evaluation_run", rid)
            self.backup(force=True)
        except BaseException as error:
            self.client.set_tag(rid, "failure", f"{type(error).__name__}: {error}"[:2000])
            self.client.set_terminated(rid, "KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            try:
                self.backup(force=True)
            except Exception as backup_error:
                print("Falló el respaldo de evaluación:", backup_error)
            raise

    @contextlib.contextmanager
    def attempt(self, config, spec, checkpoint):
        parent = self.parent(config)
        self.client.update_run(parent, status="RUNNING")
        rid = self.client.create_run(self.experiment, tags={"mlflow.runName": f"{config}/attempt_{uuid.uuid4().hex[:8]}",
            "mlflow.parentRunId": parent, "a4_run_id": self.run_id, "config": config,
            "kind": "attempt", "validation_reference": "prelabels_and_available_gold"}).info.run_id
        self.attempts[config] = rid
        try:
            (self.output / config).mkdir(parents=True, exist_ok=True)
            for key, value in spec.items():
                if not isinstance(value, (dict, list)):
                    self.client.log_param(rid, key, str(value)[:5000])
            self.client.log_param(rid, "config", config)
            self.client.log_param(rid, "resume_checkpoint", str(checkpoint))
            self.client.set_tag(rid, "drive_output", str(self.output / config))
            self.reports(config, rid)
            self.backup(force=True)
            yield rid
        except BaseException as error:
            self.client.set_tag(rid, "failure", f"{type(error).__name__}: {error}"[:2000])
            self.client.set_terminated(rid, "KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            self.client.set_terminated(parent, "FAILED")
            try:
                self.backup(force=True)
            except Exception as backup_error:
                print("Falló el respaldo de cierre:", backup_error)
            raise
        else:
            self.reports(config, rid)
            self.client.set_terminated(rid, "FINISHED")
            self.client.set_terminated(parent, "FINISHED")
            self.backup(force=True)


def commit_checkpoint(path):
    path = Path(path)
    required = ["trainer_state.json", "optimizer.pt", "scheduler.pt", "training_args.bin"]
    assert all((path / name).is_file() for name in required)
    assert list(path.glob("*safetensors")) or list(path.glob("pytorch_model*.bin"))
    assert list(path.glob("rng_state*.pth")), "Falta el estado aleatorio para reanudar"
    files = {p.name: {"sha256": sha256(p), "size": p.stat().st_size} for p in path.iterdir() if p.is_file() and p.name != "A4_COMPLETE.json"}
    save_json(path / "A4_COMPLETE.json", {"files": files})


def verified_checkpoint(directory):
    directory = Path(directory)
    candidates = sorted(directory.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]), reverse=True)
    for path in candidates:
        if not (path / "A4_COMPLETE.json").exists():
            print("Se omite checkpoint sin cierre:", path)
            continue
        files = json.loads((path / "A4_COMPLETE.json").read_text())["files"]
        if all((path / name).is_file() and (path / name).stat().st_size == meta["size"] and sha256(path / name) == meta["sha256"] for name, meta in files.items()):
            # Trainer vuelve a cargar también el mejor checkpoint previo.
            state = json.loads((path / "trainer_state.json").read_text())
            best = state.get("best_model_checkpoint")
            if best and not Path(best).is_dir():
                raise ValueError(f"Falta el mejor checkpoint requerido: {best}")
            if best and Path(best) != path:
                marker = Path(best) / "A4_COMPLETE.json"
                assert marker.is_file(), "El mejor checkpoint previo no tiene cierre"
                best_files = json.loads(marker.read_text())["files"]
                assert all((Path(best) / name).is_file() and sha256(Path(best) / name) == meta["sha256"]
                           for name, meta in best_files.items()), "Mejor checkpoint previo corrupto"
            return str(path)
        print("Se omite checkpoint con hashes inválidos:", path)
    if candidates:
        raise ValueError("Hay checkpoints pero ninguno completo. No se reinicia silenciosamente.")
    return None


class A4MLflowCallback(TrainerCallback):
    def __init__(self, tracking, run):
        self.tracking, self.run = tracking, run

    def on_log(self, args, state, control, logs=None, **kwargs):
        self.tracking.metrics(self.run, logs or {}, state.global_step)

    def on_step_end(self, args, state, control, **kwargs):
        self.tracking.backup()

    def on_save(self, args, state, control, **kwargs):
        path = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        commit_checkpoint(path)
        self.tracking.client.set_tag(self.run, "latest_complete_checkpoint", str(path))
        self.tracking.client.log_artifact(self.run, str(path / "A4_COMPLETE.json"),
            artifact_path=f"checkpoints/{path.name}")
        self.tracking.backup(force=True)


import re
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID), "RUN_ID: usa letras, números, _ y -."
tracking = A4Tracking(Path("/opt/maia/a4_tracking"), RESULTS_ROOT / "a4/_tracking/snapshots",
                      OUTPUT_ROOT, RUN_ID, SNAPSHOT_INTERVAL_SECONDS)
print("MLflow listo:", tracking.experiment, "en", mlflow.get_tracking_uri())

for config in CONFIGS:
    parent_id = tracking.parent(config)
    tracking.client.set_tag(parent_id, "isolation", "V2")
    tracking.client.set_tag(parent_id, "baseline_run_id", BASELINE_ROOT.name)
    tracking.client.set_tag(parent_id, "dataset_release", "training_v1")
    tracking.client.set_tag(parent_id, "evaluation_scope", "exploratory_validation_prelabels")


## 11. Entrenar primero base y después fragmentos

Cada configuración parte otra vez del encoder preentrenado; **fragmentos no continúa
el modelo base**. Se elige el mejor checkpoint por F1 macro en la misma validación,
con hasta cuatro épocas y parada temprana de paciencia dos. CE sin ponderar sigue
la propuesta; no se recorta Background ni se duplica la minoría. Cualquier experimento
con pesos por clase deberá usar otro RUN_ID y decidirse con val.

Los checkpoints de cada época quedan en Drive para recuperar una sesión de Colab.
`save_total_limit=2` limita espacio. Si hay OOM, bajar TRAIN_BATCH y subir GRAD_ACCUM
para mantener batch efectivo 32, con una nueva ejecución documentada.

Al terminar cada modelo se exportan encoder+cabeza, tokenizador, contrato de entrada,
política de tokens y código de inferencia. La exportación base sirve para integrar v0.
La probabilidad softmax se presenta como puntaje **sin calibrar**.


In [ ]:
import inspect
from transformers.trainer_utils import get_last_checkpoint

def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    pred = np.asarray(logits).argmax(axis=-1)
    y = prediction.label_ids
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(ID2LABEL),
                                               average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "precision_macro": p,
            "recall_macro": r, "f1_macro": f,
            "f1_micro": f1_score(y, pred, labels=list(ID2LABEL), average="micro", zero_division=0)}

INFERENCE_TEMPLATE = """import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

__ENCODER__

class CitationClassifier:
    def __init__(self, model_dir, device=None):
        self.root = Path(model_dir)
        self.policy = json.loads((self.root / "input_policy.json").read_text(encoding="utf-8"))
        self.tokenizer = AutoTokenizer.from_pretrained(self.root, local_files_only=True)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModelForSequenceClassification.from_pretrained(self.root, local_files_only=True, attn_implementation="eager").to(self.device).eval()

    def predict(self, example):
        features, stats = encode_example(example, self.policy["config"], self.tokenizer,
                                         self.policy["max_length"], self.policy["context_budget"])
        batch = {k: torch.tensor([v], device=self.device) for k, v in features.items()}
        with torch.inference_mode():
            probs = self.model(**batch).logits.float().softmax(-1)[0].cpu().tolist()
        best = max(range(len(probs)), key=probs.__getitem__)
        return {"label": self.model.config.id2label[best], "score": probs[best],
                "scores": {self.model.config.id2label[i]: p for i, p in enumerate(probs)},
                "score_calibrated": False, "token_audit": stats}
"""

# IPython conserva el código fuente de funciones definidas en sus celdas.
ENCODER_SOURCE = inspect.getsource(fair_caps) + "\n" + inspect.getsource(encode_example)
validation_results = []
for config in CONFIGS:
    workdir = OUTPUT_ROOT / config
    checkpoint_dir = workdir / "checkpoints"
    if RESUME and (workdir / "TRAINING_COMPLETE.json").exists():
        manifest = json.loads((workdir / "TRAINING_COMPLETE.json").read_text())
        for name, digest in manifest["model_hashes"].items():
            assert sha256(workdir / "model" / name) == digest
        validation_results.append(json.loads((workdir / "validation_metrics.json").read_text()))
        parent_id = tracking.parent(config)
        tracking.client.set_tag(parent_id, "drive_output", str(workdir))
        tracking.reports(config)
        tracking.client.set_terminated(parent_id, "FINISHED")
        tracking.backup(force=True)
        print(config, "ya entrenado y verificado; se omite entrenamiento.")
        continue
    last = verified_checkpoint(checkpoint_dir) if checkpoint_dir.exists() else None
    if (last or (workdir / "model/config.json").exists()) and not RESUME:
        raise ValueError(f"{config} ya tiene entrenamiento. Usa otro RUN_ID o RESUME=True.")
    with tracking.attempt(config, run_spec, last) as attempt_id:
        set_seed(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, num_labels=len(LABELS),
            id2label=ID2LABEL, label2id=LABEL2ID, problem_type="single_label_classification",
            attn_implementation="eager",
        )
        model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
        args = TrainingArguments(
            output_dir=str(checkpoint_dir), num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM, per_device_eval_batch_size=EVAL_BATCH,
            weight_decay=0.01, warmup_ratio=0.1, lr_scheduler_type="linear",
            eval_strategy="epoch", save_strategy="epoch", logging_strategy="steps", logging_steps=50,
            load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
            save_total_limit=2, save_safetensors=True, seed=SEED, data_seed=SEED,
            restore_callback_states_from_checkpoint=True,
            bf16=USE_BF16, fp16=not USE_BF16, tf32=False, full_determinism=True,
            dataloader_num_workers=2, dataloader_pin_memory=True,
            optim="adamw_torch", report_to="none", push_to_hub=False,
        )
        trainer = Trainer(model=model, args=args, train_dataset=encoded[config]["train"],
            eval_dataset=encoded[config]["validation"], processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2), A4MLflowCallback(tracking, attempt_id)])
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        train_result = trainer.train(resume_from_checkpoint=last if RESUME and last else None)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        val_metrics = trainer.evaluate()
        export_dir = workdir / "model"
        trainer.save_model(str(export_dir))
        tokenizer.save_pretrained(export_dir)
        policy = {"config": config, "max_length": MAX_LENGTH, "context_budget": CONTEXT_BUDGET,
                  "markers": MARKERS, "model_revision": MODEL_REVISION, "score_calibrated": False}
        save_json(export_dir / "input_policy.json", policy)
        save_json(export_dir / "label_mapping.json", {"label2id": LABEL2ID, "id2label": ID2LABEL})
        save_json(export_dir / "run_manifest.json", run_spec)
        (export_dir / "inference.py").write_text(INFERENCE_TEMPLATE.replace("__ENCODER__", ENCODER_SOURCE), encoding="utf-8")
        contract = {"config": config, "required_input": ["context", "title", "abstract"] if config == "base"
                    else ["context", "chunks: lista de 3 objetos {text, section}"],
                    "context_target": "Exactamente un [[TARGET]]; otros marcadores según guía rev3.1",
                    "output": {"label": "una de 9 clases", "score": "softmax sin calibrar", "scores": "9 probabilidades"}}
        save_json(export_dir / "api_contract.json", contract)
        result = {"config": config, **val_metrics, "elapsed_session_seconds": elapsed,
                  "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
                  "best_checkpoint": trainer.state.best_model_checkpoint,
                  "best_validation_macro_f1": trainer.state.best_metric,
                  "train_metrics": train_result.metrics}
        save_json(workdir / "validation_metrics.json", result)
        save_json(workdir / "training_log.json", trainer.state.log_history)
        model_hashes = {p.relative_to(export_dir).as_posix(): sha256(p)
                        for p in export_dir.rglob("*") if p.is_file()}
        save_json(workdir / "TRAINING_COMPLETE.json", {"model_hashes": model_hashes,
                  "model_dir": str(export_dir), "attempt_id": attempt_id})
        tracking.metrics(attempt_id, val_metrics, trainer.state.global_step)
        tracking.metrics(attempt_id, {"elapsed_session_seconds": elapsed,
            "peak_gpu_allocated_gib": result["peak_gpu_allocated_gib"]}, trainer.state.global_step)
        validation_results.append(result)
        print(config, "exportado en", export_dir, "F1 macro val:", val_metrics["eval_f1_macro"])
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
display(pd.DataFrame(validation_results).drop(columns=["train_metrics"]))


## 12. Comprobar la exportación y guardar predicciones de validación

La evaluación final usa **FP32 y atención eager** para ambos modelos. Se compara toda la validación
entre el checkpoint seleccionado y su exportación en la misma precisión; además,
Trainer vuelve a predecir en BF16/FP16 para medir diferencias de precisión.
Esta comprobación no entrena ni modifica los modelos existentes. Si falta el
checkpoint, se informa que la comprobación de exportación queda pendiente.

Cada ejecución crea `evaluations/v2_fp32_<id>/`, sin sobrescribir reportes anteriores,
y un hijo de tipo `evaluation` en MLflow. La selección de épocas sigue usando las
métricas originales de Trainer; `validation_fp32_*` identifica la evaluación final.
Se muestran también precisión, recall y F1 por clase y ejemplos de error por clave,
sin publicar los textos. No hay calibración de probabilidades.

Solo se procesa validación aquí. Sus matrices de confusión y métricas por clase
permiten revisar si el desbalance afecta a las minoritarias. Para un resultado final
sobre funciones de cita hace falta el gold humano de T_B.


In [ ]:
"""Evaluación del modelo exportado; se incrusta en la sección 12 del notebook."""
from pathlib import Path
from types import SimpleNamespace
import gc
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments


def probability_metrics(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=-1)
    classes = list(range(probabilities.shape[1]))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=classes, average="macro", zero_division=0)
    return {"accuracy": float(accuracy_score(labels, predictions)),
            "precision_macro": float(precision), "recall_macro": float(recall),
            "f1_macro": float(f1), "f1_micro": float(f1_score(
                labels, predictions, labels=classes, average="micro", zero_division=0))}


def compare_probabilities(reference, actual, keys, atol=2e-5, rtol=1e-4):
    reference, actual = np.asarray(reference), np.asarray(actual)
    if reference.shape != actual.shape or reference.ndim != 2 or len(keys) != len(actual) or not len(actual):
        raise ValueError("Las predicciones no tienen las mismas filas y clases")
    for values in [reference, actual]:
        if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
            raise ValueError("Probabilidades inválidas")
        if not np.allclose(values.sum(axis=1), 1, atol=1e-5):
            raise ValueError("Las probabilidades no suman uno")
    changed = reference.argmax(-1) != actual.argmax(-1)
    return {"n": len(keys), "atol": atol, "rtol": rtol,
            "probabilities_close": bool(np.allclose(reference, actual, atol=atol, rtol=rtol)),
            "max_absolute_difference": float(np.abs(reference - actual).max()),
            "label_disagreements": int(changed.sum()),
            "changed_inst_keys": [key for key, differs in zip(keys, changed) if differs]}


def predict_export_fp32(classifier, data, batch_size):
    classifier.model.float().eval()
    collator = DataCollatorWithPadding(classifier.tokenizer, pad_to_multiple_of=8)
    probabilities = []
    for start in range(0, len(data), batch_size):
        items = [{k: v for k, v in data[i].items() if k != "labels"}
                 for i in range(start, min(start + batch_size, len(data)))]
        batch = {k: v.to(classifier.device) for k, v in collator(items).items()}
        # Explícito incluso si el llamador usa un contexto autocast externo.
        with torch.inference_mode(), torch.autocast(device_type=torch.device(classifier.device).type, enabled=False):
            probabilities.extend(classifier.model(**batch).logits.float().softmax(-1).cpu().tolist())
    return np.asarray(probabilities)


def audit_export(classifier, data, keys, checkpoint, output_dir, batch_size, seed, mixed_precision):
    """Compara checkpoint/exportación en FP32 y reproduce la ruta de Trainer.

    Nunca entrena ni modifica el checkpoint o la exportación existente.
    Si falta el checkpoint, la auditoría de exportación queda explícitamente pendiente.
    """
    if mixed_precision not in {"bf16", "fp16", "fp32"}:
        raise ValueError("Precisión de Trainer no reconocida")
    if classifier.model.config._attn_implementation != "eager":
        raise ValueError("La comparación requiere atención eager en ambas rutas")
    probabilities = predict_export_fp32(classifier, data, batch_size)
    has_checkpoint = bool(checkpoint and Path(checkpoint).is_dir())
    source = Path(checkpoint) if has_checkpoint else classifier.root
    reference_model = AutoModelForSequenceClassification.from_pretrained(
        source, local_files_only=True, attn_implementation="eager").float().to(classifier.device).eval()
    try:
        audit = {"evaluation_policy": "v2_fp32", "canonical_precision": "fp32", "attention_implementation": "eager",
                 "trainer_precision": mixed_precision, "reference_source": str(source),
                 "export_check": "pending_checkpoint_missing"}
        if has_checkpoint:
            reference = SimpleNamespace(model=reference_model, tokenizer=classifier.tokenizer, device=classifier.device)
            expected = predict_export_fp32(reference, data, batch_size)
            comparison = compare_probabilities(expected, probabilities, keys)
            audit["checkpoint_vs_export_fp32"] = comparison
            if not comparison["probabilities_close"] or comparison["label_disagreements"]:
                raise ValueError(f"La exportación no reproduce el checkpoint: {comparison}")
            audit["export_check"] = "passed_full_validation"
        args = TrainingArguments(output_dir=str(Path(output_dir) / "trainer_prediction"),
            per_device_eval_batch_size=batch_size, report_to="none", seed=seed,
            use_cpu=torch.device(classifier.device).type == "cpu",
            bf16=mixed_precision == "bf16", fp16=mixed_precision == "fp16", tf32=False,
            dataloader_num_workers=0, disable_tqdm=False)
        trainer = Trainer(model=reference_model, args=args, processing_class=classifier.tokenizer,
            data_collator=DataCollatorWithPadding(classifier.tokenizer, pad_to_multiple_of=8))
        prediction = trainer.predict(data)
        logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
        trainer_probabilities = torch.as_tensor(logits).float().softmax(-1).numpy()
        audit["trainer_vs_export_fp32"] = compare_probabilities(trainer_probabilities, probabilities, keys)
        return probabilities, trainer_probabilities, audit
    finally:
        del reference_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

import importlib.util
import matplotlib.pyplot as plt

EVALUATION_ROOT = OUTPUT_ROOT / "evaluations" / ("v2_fp32_" + uuid.uuid4().hex[:12])
EVALUATION_ROOT.mkdir(parents=True, exist_ok=False)

def load_export(config):
    root = OUTPUT_ROOT / config / "model"
    spec = importlib.util.spec_from_file_location(f"a4_inference_{config}", root / "inference.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    classifier = module.CitationClassifier(root)
    # Las exportaciones de la primera corrida dejaban la atención en automático.
    # Homogeneizarla en memoria, sin modificar inference.py ni los pesos guardados.
    if classifier.model.config._attn_implementation != "eager":
        del classifier.model
        classifier.model = AutoModelForSequenceClassification.from_pretrained(
            root, local_files_only=True, attn_implementation="eager").float().to(classifier.device).eval()
    return classifier

def predict_encoded(classifier, data):
    return predict_export_fp32(classifier, data, EVAL_BATCH)

def report_predictions(config, split, rows, probs, evaluation_run):
    y = np.asarray([LABEL2ID[r["label"]] for r in rows])
    pred = probs.argmax(-1)
    report = classification_report(y, pred, labels=list(ID2LABEL), target_names=LABELS,
                                   output_dict=True, zero_division=0)
    root = EVALUATION_ROOT / config
    root.mkdir(parents=True, exist_ok=True)
    save_json(root / f"{split}_classification_report.json", report)
    metrics = {"config": config, "split": split, "n": len(rows),
               "evaluation_policy": "v2_fp32", "precision": "fp32",
               **compute_metrics(type("Prediction", (), {"predictions": probs, "label_ids": y})())}
    save_json(root / f"{split}_scores.json", metrics)
    frame = pd.DataFrame({"inst_key": [r["inst_key"] for r in rows],
                          "gold_or_prelabel": [r["label"] for r in rows],
                          "label_source": [r["label_source"] for r in rows],
                          "predicted_label": [ID2LABEL[int(x)] for x in pred]})
    for i, label in ID2LABEL.items():
        frame[f"p_{label}"] = probs[:, i]
    frame.to_csv(root / f"{split}_predictions.csv", index=False)
    frame.loc[frame.gold_or_prelabel != frame.predicted_label].to_csv(root / f"{split}_errors.csv", index=False)
    per_class = pd.DataFrame([{"label": label, **report[label]} for label in LABELS])
    per_class.to_csv(root / f"{split}_per_class.csv", index=False)
    display(per_class)
    for index, label in ID2LABEL.items():
        tracking.metrics(evaluation_run, {f"{split}/class_{index}/{key}": value
            for key, value in report[label].items()})
    cm = confusion_matrix(y, pred, labels=list(ID2LABEL))
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(root / f"{split}_confusion.csv")
    short = ["BG", "Gap", "Basis", "Comp", "App", "I/M", "Evid", "Origin", "FR"]
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set(xticks=range(9), yticks=range(9), xticklabels=short, yticklabels=short,
           xlabel="Predicción", ylabel="Referencia", title=f"{config} · {split}")
    for i in range(9):
        for j in range(9):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(root / f"{split}_confusion.png", dpi=160)
    plt.show()
    plt.close(fig)
    tracking.metrics(evaluation_run, {f"{split}_fp32_{k}": v for k, v in metrics.items()})
    tracking.metrics(tracking.parent(config), {f"{split}_fp32_{k}": v for k, v in metrics.items()})
    return metrics

val_summary = []
for config in CONFIGS:
    evaluation_dir = EVALUATION_ROOT / config
    evaluation_dir.mkdir()
    with tracking.evaluation(config, evaluation_dir) as evaluation_run:
        clf = load_export(config)
        try:
            sample = clf.predict(texts[val_rows[0]["inst_key"]])
            single = predict_encoded(clf, encoded[config]["validation"].select([0]))[0]
            assert np.allclose(single, [sample["scores"][k] for k in LABELS], atol=2e-5)
            training_metrics = json.loads((OUTPUT_ROOT / config / "validation_metrics.json").read_text())
            precision = "bf16" if USE_BF16 else "fp16"
            probs, trainer_probs, audit = audit_export(clf, encoded[config]["validation"],
                [r["inst_key"] for r in val_rows], training_metrics.get("best_checkpoint"),
                evaluation_dir, EVAL_BATCH, SEED, precision)
            replayed = probability_metrics([LABEL2ID[r["label"]] for r in val_rows], trainer_probs)
            audit.update({"run_id": RUN_ID, "config": config, "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION, "split_hashes": split_hashes, "labels": LABELS,
                "label_sources": dict(collections.Counter(r["label_source"] for r in val_rows)),
                "training_complete_sha256": sha256(OUTPUT_ROOT / config / "TRAINING_COMPLETE.json"),
                "recorded_training_f1_macro": training_metrics.get("eval_f1_macro"),
                "replayed_trainer_metrics": replayed})
            save_json(evaluation_dir / "export_precision_audit.json", audit)
            tracking.client.set_tag(evaluation_run, "export_check", audit["export_check"])
            tracking.client.set_tag(evaluation_run, "trainer_precision", precision)
            tracking.client.set_tag(evaluation_run, "selected_checkpoint", str(training_metrics.get("best_checkpoint")))
            tracking.client.set_tag(evaluation_run, "model_revision", MODEL_REVISION)
            tracking.metrics(evaluation_run, {f"validation_trainer_replay_{k}": v for k, v in replayed.items()})
            tracking.metrics(evaluation_run, {"precision_label_disagreements": audit["trainer_vs_export_fp32"]["label_disagreements"]})
            val_summary.append(report_predictions(config, "validation", val_rows, probs, evaluation_run))
            print("Auditoría:", audit["export_check"], "cambios de etiqueta por precisión:",
                  audit["trainer_vs_export_fp32"]["label_disagreements"])
        finally:
            del clf
            gc.collect()
            torch.cuda.empty_cache()
pd.DataFrame(val_summary).to_csv(EVALUATION_ROOT / "validation_comparison.csv", index=False)
if {"base", "fragmentos"} <= {row["config"] for row in val_summary}:
    scores = {row["config"]: row["f1_macro"] for row in val_summary}
    print("Delta F1 macro de validación (fragmentos - base):", scores["fragmentos"] - scores["base"])
print("Evaluación versionada:", EVALUATION_ROOT)
display(pd.DataFrame(val_summary))


## 12.1. Comparar V2 con las referencias V0 congeladas

Las validaciones propias tienen tamaños distintos. La comparación principal de esta
prueba usa sus citas comunes: exige que toda val V2 esté en val V0, alinea por clave
y verifica las etiquetas antes de calcular métricas. Se muestran las dos entradas,
las categorías escasas, una gráfica y CSV exportables. No se compara con T_B ni se
ajusta nada después de sus resultados. El tamaño de train también cambia.


In [ ]:

reference_counts = pd.read_csv(BASELINE_ROOT / "split_class_counts.csv", index_col=0)
own_validation, common_validation, common_per_class = compare_v0_v2(
    BASELINE_EVALUATION_ROOT, EVALUATION_ROOT, BASELINE_ROOT.name, RUN_ID, LABELS,
    int(reference_counts["train"].sum()), len(train_rows), EVALUATION_ROOT)
print("Validaciones propias: tamaños distintos; comparación descriptiva.")
display(own_validation[["isolation", "config", "train_n", "n", "f1_macro", "accuracy"]])
print("Mismas citas de validación, alineadas por inst_key; no es el test humano.")
display(common_validation[["isolation", "config", "train_n", "n", "f1_macro", "accuracy"]])
display(common_per_class.loc[common_per_class.label.isin(["Evidence", "Improvement/Modification"]),
    ["isolation", "config", "label", "f1_class", "recall_class", "support"]])
common_scores = common_validation.pivot(index="config", columns="isolation", values="f1_macro")
ax = common_scores.reindex(["base", "fragmentos"])[["V0", "V2"]].plot.bar(
    figsize=(8, 4), ylim=(0, 1), rot=0, color=["#8fa6be", "#16877c"])
ax.set(xlabel="Configuración de entrada", ylabel="F1 macro FP32",
       title="V0 frente a V2 · mismas citas de validación")
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3)
ax.figure.tight_layout()
ax.figure.savefig(EVALUATION_ROOT / "isolation_common_f1.png", dpi=160)
plt.show()
plt.close(ax.figure)
for config in CONFIGS:
    current = common_validation.loc[(common_validation.config == config) &
                                    (common_validation.isolation == "V2")].iloc[0]
    reference = common_validation.loc[(common_validation.config == config) &
                                      (common_validation.isolation == "V0")].iloc[0]
    delta = float(current.f1_macro - reference.f1_macro)
    print(config, "delta F1 macro V2 - V0 en validación común:", delta)
    tracking.metrics(tracking.parent(config), {
        "common_validation_n": int(current.n), "common_validation_v2_f1_macro": float(current.f1_macro),
        "common_validation_v0_f1_macro": float(reference.f1_macro), "common_validation_delta_f1_macro": delta})
save_json(EVALUATION_ROOT / "isolation_comparison_notes.json", {
    "baseline_run_id": BASELINE_ROOT.name, "run_id": RUN_ID, "test_used": False,
    "precision": "fp32", "common_validation_n": int(common_validation.n.iloc[0]),
    "interpretation": "Cambian aislamiento y tamaño de train; una semilla, preetiquetas A2.",
    "common_keys_sha256": sha256(EVALUATION_ROOT / "isolation_common_keys.csv")})


## 13. Evaluación final opcional sobre T_B humano

Esta sección se omite con la configuración inicial. Para la evaluación final, preparar
**una nueva ejecución** con el gold consolidado, `GOLD_FINAL=True` y `RUN_TEST=True`,
congelando antes variante, política de tokens e hiperparámetros mediante validación.
Debe haber al menos 300 referencias humanas válidas por clase; `unclear` y paquetes
sin gold no entran. No se sustituyen por la pre-etiqueta ni se muestrean 300 a conveniencia.

Se exportan métricas macro/micro, métricas por clase, matrices y predicciones.
Con V0, se desglosa el desempeño según si el **citado** fue visto en train.
Un puntaje cero con un subgrupo vacío no se reporta: se deja sin resultado.


In [ ]:
test_summary = []
test_probabilities = {}
if not RUN_TEST:
    print("Evaluación final pendiente de gold T_B consolidado. El entrenamiento puede terminar aquí.")
else:
    train_cited = {r["cited_paper_id"] for r in train_rows}
    for config in CONFIGS:
        with tracking.evaluation(config, EVALUATION_ROOT / config) as test_run:
            clf = load_export(config)
            probs = predict_encoded(clf, encoded[config]["test"])
            test_probabilities[config] = probs
            test_summary.append(report_predictions(config, "test", test_rows, probs, test_run))
            subgroup_metrics = {}
            for seen in (True, False):
                idx = [i for i, r in enumerate(test_rows) if (r["cited_paper_id"] in train_cited) == seen]
                if not idx:
                    subgroup_metrics[str(seen)] = {"n": 0, "f1_macro": None}
                    continue
                y = np.array([LABEL2ID[test_rows[i]["label"]] for i in idx])
                pred = probs[idx].argmax(-1)
                subgroup_metrics[str(seen)] = {"n": len(idx), "f1_macro": f1_score(
                    y, pred, labels=list(ID2LABEL), average="macro", zero_division=0)}
            save_json(EVALUATION_ROOT / config / "test_cited_seen_unseen.json", subgroup_metrics)
            del clf
            gc.collect()
            torch.cuda.empty_cache()
    display(pd.DataFrame(test_summary))
    pd.DataFrame(test_summary).to_csv(EVALUATION_ROOT / "test_comparison.csv", index=False)


## 14. Comparar el aporte de Top-3 y empaquetar los artefactos

El resultado principal será `F1_macro(fragmentos) - F1_macro(base)` en el mismo T_B.
Se calcula un intervalo bootstrap pareado por **artículo citante**, manteniendo juntas
sus instancias. Es un análisis exploratorio de esta semilla: para sostener una mejora
consistente conviene repetir ambas configuraciones con varias semillas, con split fijo
y sin seleccionar por test. Esta celda no modifica modelos después de ver el test.

Los ZIP contienen pesos, tokenizador, política y módulo de inferencia; no el corpus.
Para integrar: extraer, importar `CitationClassifier` de `inference.py` y llamar
`predict(example)` con el contrato de entrada. Hay nueve puntajes softmax sin calibrar.
Los CSV con claves/etiquetas y el manifiesto quedan fuera de los ZIP de modelos.

Referencias técnicas:
[SciBERT](https://aclanthology.org/D19-1371/),
[modelo publicado](https://huggingface.co/allenai/scibert_scivocab_uncased),
[Trainer 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer),
[MultiCite](https://aclanthology.org/2022.naacl-main.137/).


In [ ]:
if RUN_TEST and {"base", "fragmentos"} <= test_probabilities.keys():
    y = np.asarray([LABEL2ID[r["label"]] for r in test_rows])
    p0 = test_probabilities["base"].argmax(-1)
    p1 = test_probabilities["fragmentos"].argmax(-1)
    by_citing = collections.defaultdict(list)
    for i, row in enumerate(test_rows):
        by_citing[row["citing_paper_id"]].append(i)
    groups = list(by_citing.values())
    rng = np.random.default_rng(SEED)
    score = lambda truth, pred: f1_score(truth, pred, labels=list(ID2LABEL), average="macro", zero_division=0)
    deltas = []
    for _ in range(1000):
        sampled = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in sampled])
        deltas.append(score(y[idx], p1[idx]) - score(y[idx], p0[idx]))
    comparison = {"delta_macro_f1": score(y, p1) - score(y, p0),
                  "cluster_bootstrap_95_ci": np.quantile(deltas, [0.025, 0.975]).tolist(),
                  "cluster_unit": "citing_paper_id", "replicates": 1000, "seed": SEED}
    save_json(EVALUATION_ROOT / "paired_test_comparison.json", comparison)
    print(comparison)
for config in CONFIGS:
    model_dir = OUTPUT_ROOT / config / "model"
    archive = OUTPUT_ROOT / f"scibert_g15_{config}.zip"
    if not archive.exists():
        archive = shutil.make_archive(str(archive.with_suffix("")), "zip", model_dir)
    print("Modelo:", archive)
print("Artefactos privados:", OUTPUT_ROOT)
for config in CONFIGS:
    tracking.reports(config)
    tracking.client.set_tag(tracking.parent(config), "model_zip_drive", str(OUTPUT_ROOT / f"scibert_g15_{config}.zip"))
    tracking.client.set_tag(tracking.parent(config), "latest_evaluation_drive", str(EVALUATION_ROOT / config))
    for path in sorted(EVALUATION_ROOT.glob("*.csv")) + sorted(EVALUATION_ROOT.glob("*.json")) + sorted(EVALUATION_ROOT.glob("*.png")):
        tracking.client.log_artifact(tracking.parent(config), str(path), artifact_path="evaluation_comparisons/" + EVALUATION_ROOT.name)
tracking.backup(force=True)
print("A4 entrenado; la evaluación final solo está completa si RUN_TEST=True y el gold pasó los controles.")
